In [ ]:
import pickle
import numpy as np
import pandas as pd

In [ ]:
# Loop through all three batches
for batch_num in [1, 2, 3]:
    print(f"BATCH {batch_num}")
    
    # Load the batch
    bat_dict = pickle.load(open(f'batch{batch_num}.pkl', 'rb'))
    
    # Quick overview
    print(f"Total batteries: {len(bat_dict)}")
    print(f"\nBattery names: {list(bat_dict.keys())[:]}")
    
    # Look at one battery
    first_battery_name = list(bat_dict.keys())[0]
    sample_battery = bat_dict[first_battery_name]
    print(f"\nStructure of {first_battery_name}:")
    print(f"Keys: {sample_battery.keys()}\n")

In [ ]:
# Load all batches
batch1 = pickle.load(open('batch1.pkl', 'rb'))
batch2 = pickle.load(open('batch2.pkl', 'rb'))
batch3 = pickle.load(open('batch3.pkl', 'rb'))

In [ ]:
# Merge all batches
bat_dict = {**batch1, **batch2, **batch3}

In [ ]:
# Count total before filtering
total_batteries = len(bat_dict)

In [ ]:
# Extract cycle life for ALL batteries
cycle_lives = []
battery_names = []
batch_nums = []
bad_batteries = []

for bat_name, bat_data in bat_dict.items():
    cycle_life = bat_data['cycle_life']
    
    # Flatten the value regardless of format
    if isinstance(cycle_life, (list, np.ndarray)):
        cycle_life = float(np.array(cycle_life).flatten()[0])
    else:
        cycle_life = float(cycle_life)
    
    # Check if value is NaN or invalid
    if np.isnan(cycle_life) or cycle_life <= 0:
        bad_batteries.append(bat_name)
        continue
    
    cycle_lives.append(cycle_life)
    battery_names.append(bat_name)
    
    # Extract batch number from battery name
    batch_num = int(bat_name[1])
    batch_nums.append(batch_num)

In [ ]:
# Create DataFrame with only valid batteries
merged_df = pd.DataFrame({
    'Battery': battery_names,
    'Batch': batch_nums,
    'Cycle_Life': cycle_lives
})

In [ ]:
# Report filtered batteries
print(f"Filtered out {len(bad_batteries)} batteries with NaN/invalid values:")
print(f"   {bad_batteries}\n")

In [ ]:
# Analyze by batch
for batch_num in [1, 2, 3]:
    batch_df = merged_df[merged_df['Batch'] == batch_num].sort_values('Cycle_Life', ascending=False)
    
    if len(batch_df) == 0:
        print(f"\nNo valid data for batch #{batch_num}")
        continue
    
    # Print statistics
    print(f"\n{'='*60}")
    print(f"BATCH #{batch_num}")
    print(f"{'='*60}")
    print(f"Total batteries: {len(batch_df)}")
    print(f"Max cycle life: {batch_df['Cycle_Life'].max():.0f} ({batch_df.iloc[0]['Battery']})")
    print(f"Min cycle life: {batch_df['Cycle_Life'].min():.0f} ({batch_df.iloc[-1]['Battery']})")
    print(f"Mean cycle life: {batch_df['Cycle_Life'].mean():.0f}")
    print(f"Median cycle life: {batch_df['Cycle_Life'].median():.0f}")
    print(f"Std deviation: {batch_df['Cycle_Life'].std():.0f}")
    
    # Show top 5 batteries
    print(f"\nTop 5 longest-lasting:")
    print(batch_df.head(5).to_string(index=False))
    
    # Show bottom 5 batteries
    print(f"\nTop 5 shortest-lasting:")
    print(batch_df.tail(5).to_string(index=False))

# Overall statistics
print(f"\n{'='*60}")
print(f"OVERALL (ALL BATCHES)")
print(f"{'='*60}")
print(f"Total valid batteries: {len(merged_df)} out of {total_batteries}")
print(f"Max cycle life: {merged_df['Cycle_Life'].max():.0f}")
print(f"Min cycle life: {merged_df['Cycle_Life'].min():.0f}")
print(f"Mean cycle life: {merged_df['Cycle_Life'].mean():.0f}")